# Tarea 2 · Parte 2: evaluación y extracción estructurada

Todos los modelos se evalúan con la misma **muestra de evaluación** de la Parte 1 (sección 8 de `01_extraccion.ipynb`):
10 documentos (3 impresos, 3 fotos y 4 manuscritos) y 10 clips (8 entrevistas y 2 noticieros de archivo), transcritos a
mano en `datos/raw/referencias/`.

- **Métricas**: CER para documentos y WER para audio, con `jiwer`, sobre textos normalizados con un mismo criterio
  (sección 1). En audio también se mide el factor de tiempo real (RTF = tiempo de proceso / duración del audio).
- **Textos sin seudonimizar**: las métricas se calculan sobre las referencias y salidas originales, que quedan en
  `datos/raw/` (fuera de Git). Así los marcadores `[PERSONA_n]` no cuentan como errores. Este notebook solo muestra
  textos ya seudonimizados.
- **Configuraciones fijadas antes de medir**: la celda siguiente define todos los modelos y sus parámetros y los guarda
  en `modelos/config_evaluacion.json` antes de calcular cualquier métrica.
- **Caché**: cada salida se guarda en `datos/raw/evaluacion/<modelo>/<id>.json` con su tiempo. Si existe, no se recalcula.

In [1]:
import gc
import json
import os
import re
import sys
import time
import unicodedata
from datetime import date
from pathlib import Path

import torch  # antes que paddle: si paddle carga primero, torch no encuentra sus DLL de CUDA

for carpeta in Path(sys.prefix, "Lib", "site-packages", "nvidia").glob("*/bin"):
    os.add_dll_directory(str(carpeta))

import jiwer
import numpy as np
import pandas as pd
import plotly.express as px
import soundfile as sf
from IPython.display import display

DATOS = Path("datos")
RAW = DATOS / "raw"
EVAL = RAW / "evaluacion"
EVAL.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 80)
px.defaults.template = "plotly_white"

documentos = pd.read_csv(DATOS / "documentos.csv", keep_default_na=False)
transcripciones = pd.read_csv(DATOS / "transcripciones.csv", keep_default_na=False)
fuentes = pd.read_csv(DATOS / "fuentes.csv", keep_default_na=False)
muestra_docs = documentos[documentos.texto_referencia != ""].merge(fuentes[["id", "autor"]], on="id")
muestra_clips = transcripciones[transcripciones.texto_referencia != ""].merge(fuentes[["id", "tipo", "autor"]], on="id")

# Condición de grabación de cada clip de la muestra
CONDICION = {"clip_004": "entrevista, exterior", "clip_005": "entrevista, manifestación",
             "clip_009": "entrevista, calle (Cusco)", "clip_011": "entrevista, calle", "clip_016": "entrevista, feria",
             "clip_018": "entrevista, centro comercial", "clip_019": "entrevista, centro comercial",
             "clip_022": "entrevista, interior", "clip_023": "noticiero de archivo con música",
             "clip_025": "noticiero de archivo con música"}
muestra_clips["condicion"] = muestra_clips.id.map(CONDICION)


def fin_ventana(clip_id, limite=60):
    """Final del segmento de Whisper large-v3 que cruza el segundo 60: ahí terminan las referencias."""
    seg = pd.read_csv(RAW / "audio" / "salidas" / f"{clip_id}.csv")
    return float(seg[seg.inicio < limite].fin.max())


# Cinco clips los cortó a mano quien transcribió, y su referencia no termina justo donde termina ese segmento.
# Para ellos la ventana llega al final del segmento de Whisper large-v3 que contiene las últimas palabras de la
# referencia ("genial" en clip_009 empieza en 59.4 s). Corrección detectada al revisar la línea base: se aplica igual
# a todos los modelos.
FIN_REFERENCIA = {"clip_009": 60.0, "clip_011": 57.48, "clip_016": 61.96, "clip_018": 65.42}
muestra_clips["fin_seg"] = [FIN_REFERENCIA.get(c, fin_ventana(c)) for c in muestra_clips.id]
SR = 16_000
audios = {c: sf.read(RAW / "audio" / f"{c}.wav", dtype="float32")[0][: int(f * SR)]
          for c, f in zip(muestra_clips.id, muestra_clips.fin_seg)}
referencias = {i: (RAW / "referencias" / f"{i}.txt").read_text(encoding="utf-8")
               for i in list(muestra_docs.id) + list(muestra_clips.id)}

display(muestra_docs[["id", "tipo"]])
display(muestra_clips[["id", "tipo", "condicion", "fin_seg"]])

,id,tipo
0,doc_001,impreso
1,doc_007,impreso
2,doc_008,impreso
3,doc_011,foto
4,doc_013,foto
5,doc_016,foto
6,doc_018,manuscrito
7,doc_019,manuscrito
8,doc_021,manuscrito
9,doc_023,manuscrito


,id,tipo,condicion,fin_seg
0,clip_004,entrevista,"entrevista, exterior",66.24
1,clip_005,entrevista,"entrevista, manifestación",61.22
2,clip_009,entrevista,"entrevista, calle (Cusco)",60.00
3,clip_011,entrevista,"entrevista, calle",57.48
4,clip_016,entrevista,"entrevista, feria",61.96
5,clip_018,entrevista,"entrevista, centro comercial",65.42
6,clip_019,entrevista,"entrevista, centro comercial",61.18
7,clip_022,entrevista,"entrevista, interior",66.04
8,clip_023,noticiero,noticiero de archivo con música,64.18
9,clip_025,noticiero,noticiero de archivo con música,65.37


In [2]:
CONFIGS = {
    "ocr": {
        "paddle_base": {"descripcion": "Línea base: PaddleOCR sin personalización", "modelo": "PaddleOCR 3.7.0 (PP-OCRv6 medium)",
                        "parametros": {"lang": "es"}, "dispositivo": "cpu"},
        "paddle_parte1": {"descripcion": "PaddleOCR de la Parte 1: orientación, enderezado y CLAHE en fotos",
                          "modelo": "PaddleOCR 3.7.0 (PP-OCRv6 medium)",
                          "parametros": {"lang": "es", "use_doc_orientation_classify": True, "use_doc_unwarping": True,
                                         "use_textline_orientation": True, "clahe_fotos": True},
                          "dispositivo": "cpu"},
        "easyocr": {"descripcion": "Alternativa: EasyOCR (detector CRAFT + reconocedor CRNN), otra arquitectura que PaddleOCR",
                    "modelo": "EasyOCR 1.7.2", "parametros": {"lang_list": ["es"], "paragraph": False}, "dispositivo": "cuda"},
    },
    "asr": {
        "whisper_small_cpu": {"descripcion": "Línea base: faster-whisper small en CPU, sin ajustes",
                              "modelo": "faster-whisper small", "parametros": {"compute_type": "int8", "beam_size": 5},
                              "dispositivo": "cpu"},
        "whisper_large_gpu": {"descripcion": "faster-whisper large-v3 de la Parte 1",
                              "modelo": "faster-whisper large-v3",
                              "parametros": {"compute_type": "float16", "beam_size": 5, "language": "es", "vad_filter": True,
                                             "min_silence_duration_ms": 500},
                              "dispositivo": "cuda"},
        "qwen3_asr": {"descripcion": "Alternativa: Qwen3-ASR 0.6B, tramos de voz de hasta 25 s cortados con el VAD Silero",
                      "modelo": "Qwen/Qwen3-ASR-0.6B-hf", "parametros": {"dtype": "bfloat16", "language": "es", "max_tramo_s": 25},
                      "dispositivo": "cuda"},
    },
    "llm": {"modelo": "Qwen/Qwen2.5-3B-Instruct", "parametros": {"dtype": "bfloat16", "temperature": 0, "max_new_tokens": 512},
            "dispositivo": "cuda"},
    "normalizacion": "minúsculas, NFC, sin marcas entre corchetes, sin guiones de corte de línea, sin puntuación, espacios simples",
    "ventana_audio": {"criterio": "de 0 al final del segmento de Whisper large-v3 que contiene las últimas palabras de la referencia",
                      "segundos": dict(zip(muestra_clips.id, muestra_clips.fin_seg))},
    "muestra": {"documentos": muestra_docs.id.tolist(), "clips": muestra_clips.id.tolist()},
}
json.dump(CONFIGS, open("modelos/config_evaluacion.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Configuraciones guardadas en modelos/config_evaluacion.json antes de medir.")

Configuraciones guardadas en modelos/config_evaluacion.json antes de medir.


## 1. Normalización

Referencias y salidas automáticas pasan por la misma función antes de medir:

1. Unicode NFC y minúsculas. Se conservan las tildes y la ortografía histórica ("exércitos", "jeneral"), porque leerlas
   bien es parte de la tarea del OCR.
2. Se quitan las marcas entre corchetes que no son texto ni palabras dichas: `[HABLANTE_1]`, `[ilegible]`,
   `[inaudible]`, `[risas]`.
3. Se unen las palabras cortadas al final de línea ("Ma- riscal" → "mariscal").
4. Se quita la puntuación y se dejan espacios simples.

El CER se calcula sobre caracteres y el WER sobre palabras. Cada tabla muestra el valor por caso y su promedio.

In [3]:
def normalizar(texto):
    t = unicodedata.normalize("NFC", texto).lower()
    t = re.sub(r"\[[^\]]*\]", " ", t)
    t = re.sub(r"(\w)-\s+(\w)", r"\1\2", t)
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def cer(ref, hip):
    return jiwer.cer(normalizar(ref), normalizar(hip) or " ")


def wer(ref, hip):
    return jiwer.wer(normalizar(ref), normalizar(hip) or " ")


def correr(modelo, casos, funcion):
    """Ejecuta `funcion(caso) -> texto` para cada caso que no esté en caché y guarda el texto y los segundos."""
    (EVAL / modelo).mkdir(exist_ok=True)
    salida = {}
    for caso in casos:
        f = EVAL / modelo / f"{caso}.json"
        if not f.exists():
            t0 = time.perf_counter()
            texto = funcion(caso)
            json.dump({"texto": texto, "segundos": time.perf_counter() - t0}, open(f, "w", encoding="utf-8"), ensure_ascii=False)
        salida[caso] = json.load(open(f, encoding="utf-8"))
    return salida


print(normalizar("[HABLANTE_1] Ma- riscal de Campo, ¿no? [inaudible] EXÉRCITOS."))

mariscal de campo no exércitos


## 2. Línea base sin ajuste

- **Documentos**: PaddleOCR con solo `lang="es"`, sobre la imagen original y sin preprocesamiento.
- **Audio**: faster-whisper `small` en CPU (`int8`), con los valores por defecto (idioma detectado, sin VAD).

PaddleOCR 3.7 ya trae activadas por defecto la clasificación de orientación, el enderezado de página (UVDoc) y la
orientación de líneas, y usa los modelos PP-OCRv6 medium. Por eso la línea base solo se diferencia de la configuración
de la Parte 1 en el CLAHE que esta aplica a las fotos.

In [4]:
import paddleocr

ocr_base = None


def leer_paddle_base(doc_id):
    global ocr_base
    if ocr_base is None:
        ocr_base = paddleocr.PaddleOCR(lang="es")
    archivo = muestra_docs.set_index("id").loc[doc_id, "archivo"]
    return " ".join(ocr_base.predict(archivo)[0]["rec_texts"])


res_paddle_base = correr("paddle_base", muestra_docs.id, leer_paddle_base)
del ocr_base
_ = gc.collect()

C:\Users\victo\Documents\GitHub\qlabpucp_python_intermedio\tarea_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from faster_whisper import WhisperModel

small = None


def transcribir_small(clip_id):
    global small
    if small is None:
        small = WhisperModel("small", device="cpu", compute_type="int8")
    segs, _ = small.transcribe(audios[clip_id], beam_size=5)
    return " ".join(s.text.strip() for s in segs)


res_small = correr("whisper_small_cpu", muestra_clips.id, transcribir_small)
del small
_ = gc.collect()

In [6]:
base_docs = muestra_docs[["id", "tipo"]].assign(cer=[cer(referencias[i], res_paddle_base[i]["texto"]) for i in muestra_docs.id])
base_clips = muestra_clips[["id", "condicion", "fin_seg"]].assign(
    wer=[wer(referencias[i], res_small[i]["texto"]) for i in muestra_clips.id],
    rtf=[res_small[i]["segundos"] / f for i, f in zip(muestra_clips.id, muestra_clips.fin_seg)])
print(f"Línea base · CER medio (PaddleOCR): {base_docs.cer.mean():.3f} · WER medio (faster-whisper small, CPU): "
      f"{base_clips.wer.mean():.3f} · RTF medio: {base_clips.rtf.mean():.2f}")
display(base_docs.round(3))
display(base_clips.round(3))

Línea base · CER medio (PaddleOCR): 0.255 · WER medio (faster-whisper small, CPU): 0.316 · RTF medio: 0.28


,id,tipo,cer
0,doc_001,impreso,0.041
1,doc_007,impreso,0.060
2,doc_008,impreso,0.062
3,doc_011,foto,0.013
4,doc_013,foto,0.815
5,doc_016,foto,0.063
6,doc_018,manuscrito,0.498
7,doc_019,manuscrito,0.354
8,doc_021,manuscrito,0.300
9,doc_023,manuscrito,0.347


,id,condicion,fin_seg,wer,rtf
0,clip_004,"entrevista, exterior",66.24,0.431,0.481
1,clip_005,"entrevista, manifestación",61.22,0.743,0.219
2,clip_009,"entrevista, calle (Cusco)",60.00,0.438,0.144
3,clip_011,"entrevista, calle",57.48,0.318,0.161
4,clip_016,"entrevista, feria",61.96,0.281,0.147
5,clip_018,"entrevista, centro comercial",65.42,0.180,0.126
6,clip_019,"entrevista, centro comercial",61.18,0.083,0.316
7,clip_022,"entrevista, interior",66.04,0.315,0.354
8,clip_023,noticiero de archivo con música,64.18,0.360,0.660
9,clip_025,noticiero de archivo con música,65.37,0.014,0.239


## 3. Modelos mejorados y alternativos

**Documentos**

- **PaddleOCR de la Parte 1**: misma red, pero con clasificación de orientación, enderezado de página y CLAHE en las
  fotos. Se usa la salida ya guardada en `datos/raw/documentos/ocr/`.
- **EasyOCR** (alternativa): detector CRAFT y reconocedor CRNN, una arquitectura distinta de la de PaddleOCR, con modelo
  para español. Corre en GPU. Tesseract no está instalado en este equipo y EasyOCR se instala con pip.

**Audio**

- **faster-whisper large-v3 en GPU**: la configuración de la Parte 1 (idioma español, VAD y `float16`).
- **Qwen3-ASR 0.6B** (alternativa): modelo abierto de Alibaba que soporta español. Acepta hasta ~25 s por llamada, así
  que el audio se corta en tramos de voz con el mismo VAD Silero de faster-whisper.

In [7]:
res_paddle_p1 = {i: {"texto": " ".join(json.load(open(RAW / "documentos" / "ocr" / f"{i}.json", encoding="utf-8"))["rec_texts"]),
                     "segundos": None} for i in muestra_docs.id}

import easyocr

lector = None


def leer_easyocr(doc_id):
    global lector
    if lector is None:
        lector = easyocr.Reader(["es"], gpu=True, verbose=False)
    return " ".join(lector.readtext(muestra_docs.set_index("id").loc[doc_id, "archivo"], detail=0, paragraph=False))


res_easyocr = correr("easyocr", muestra_docs.id, leer_easyocr)
del lector
_ = gc.collect()
torch.cuda.empty_cache()

In [8]:
large = None


def transcribir_large(clip_id):
    global large
    if large is None:
        large = WhisperModel("large-v3", device="cuda", compute_type="float16")
    segs, _ = large.transcribe(audios[clip_id], beam_size=5, language="es", vad_filter=True,
                               vad_parameters={"min_silence_duration_ms": 500})
    return " ".join(s.text.strip() for s in segs)


res_large = correr("whisper_large_gpu", muestra_clips.id, transcribir_large)
del large
_ = gc.collect()
torch.cuda.empty_cache()

In [9]:
from faster_whisper.vad import VadOptions, get_speech_timestamps
from transformers import AutoProcessor, Qwen3ASRForConditionalGeneration

qwen = None


def transcribir_qwen(clip_id):
    global qwen, proc
    if qwen is None:
        proc = AutoProcessor.from_pretrained("Qwen/Qwen3-ASR-0.6B-hf")
        qwen = Qwen3ASRForConditionalGeneration.from_pretrained("Qwen/Qwen3-ASR-0.6B-hf", dtype=torch.bfloat16).to("cuda")
    audio = audios[clip_id]
    tramos = get_speech_timestamps(audio, VadOptions(min_silence_duration_ms=500, max_speech_duration_s=25))
    textos = []
    with torch.inference_mode():
        for t in tramos:
            entrada = proc.apply_transcription_request(audio[t["start"]:t["end"]], language="es").to("cuda", dtype=torch.bfloat16)
            salida = qwen.generate(**entrada, max_new_tokens=448)
            textos.append(proc.decode(salida[0, entrada["input_ids"].shape[1]:], return_format="transcription_only").strip())
    return " ".join(textos)


res_qwen = correr("qwen3_asr", muestra_clips.id, transcribir_qwen)
del qwen
_ = gc.collect()
torch.cuda.empty_cache()

## 4. Comparativa

CER por documento y WER por clip para todos los modelos, con el tiempo de proceso. En audio, el RTF menor que 1 indica
que el modelo transcribe más rápido que la duración del audio.

In [10]:
OCR = {"PaddleOCR base (CPU)": res_paddle_base, "PaddleOCR Parte 1 (CPU)": res_paddle_p1, "EasyOCR (GPU)": res_easyocr}
ASR = {"faster-whisper small (CPU, base)": res_small, "faster-whisper large-v3 (GPU)": res_large, "Qwen3-ASR 0.6B (GPU)": res_qwen}

cer_docs = pd.DataFrame([{"id": i, "tipo": t, "modelo": m, "cer": cer(referencias[i], r[i]["texto"]), "segundos": r[i]["segundos"]}
                         for m, r in OCR.items() for i, t in zip(muestra_docs.id, muestra_docs.tipo)])
wer_clips = pd.DataFrame([{"id": i, "condicion": c, "modelo": m, "wer": wer(referencias[i], r[i]["texto"]),
                           "rtf": r[i]["segundos"] / f} for m, r in ASR.items()
                          for i, c, f in zip(muestra_clips.id, muestra_clips.condicion, muestra_clips.fin_seg)])

resumen_ocr = cer_docs.pivot_table(index="modelo", columns="tipo", values="cer", aggfunc="mean")
resumen_ocr["CER medio"] = cer_docs.groupby("modelo").cer.mean()
resumen_ocr["segundos por página"] = cer_docs.groupby("modelo").segundos.mean()
resumen_asr = wer_clips.groupby("modelo").agg(**{"WER medio": ("wer", "mean"), "WER mediano": ("wer", "median"),
                                                 "RTF medio": ("rtf", "mean")})
display(resumen_ocr.round(3))
display(resumen_asr.round(3))

tipo,foto,impreso,manuscrito,CER medio,segundos por página
modelo,,,,,
EasyOCR (GPU),0.255,0.206,0.751,0.439,2.088
PaddleOCR Parte 1 (CPU),0.522,0.055,0.375,0.323,NaN
PaddleOCR base (CPU),0.297,0.055,0.375,0.255,29.365


,WER medio,WER mediano,RTF medio
modelo,,,
Qwen3-ASR 0.6B (GPU),0.237,0.239,0.298
faster-whisper large-v3 (GPU),0.189,0.212,0.081
"faster-whisper small (CPU, base)",0.316,0.317,0.285


In [11]:
display(cer_docs.pivot_table(index=["tipo", "id"], columns="modelo", values="cer").round(3))
display(wer_clips.pivot_table(index=["condicion", "id"], columns="modelo", values="wer").round(3))

modelo              EasyOCR (GPU)  PaddleOCR Parte 1 (CPU)  \
tipo       id                                                
foto       doc_011          0.061                    0.176   
           doc_013          0.452                    0.694   
           doc_016          0.253                    0.696   
impreso    doc_001          0.266                    0.041   
           doc_007          0.205                    0.060   
           doc_008          0.147                    0.062   
manuscrito doc_018          0.770                    0.498   
           doc_019          0.766                    0.354   
           doc_021          0.633                    0.300   
           doc_023          0.835                    0.347   

modelo              PaddleOCR base (CPU)  
tipo       id                             
foto       doc_011                 0.013  
           doc_013                 0.815  
           doc_016                 0.063  
impreso    doc_001                 0.041  
           doc_007                 0.060  
           doc_008                 0.062  
manuscrito doc_018                 0.498  
           doc_019                 0.354  
           doc_021                 0.300  
           doc_023                 0.347

modelo                                    Qwen3-ASR 0.6B (GPU)  \
condicion                       id                               
entrevista, calle               clip_011                 0.209   
entrevista, calle (Cusco)       clip_009                 0.371   
entrevista, centro comercial    clip_018                 0.286   
                                clip_019                 0.100   
entrevista, exterior            clip_004                 0.363   
entrevista, feria               clip_016                 0.194   
entrevista, interior            clip_022                 0.268   
entrevista, manifestación       clip_005                 0.336   
noticiero de archivo con música clip_023                 0.146   
                                clip_025                 0.099   

modelo                                    faster-whisper large-v3 (GPU)  \
condicion                       id                                        
entrevista, calle               clip_011                          0.264   
entrevista, calle (Cusco)       clip_009                          0.286   
entrevista, centro comercial    clip_018                          0.188   
                                clip_019                          0.075   
entrevista, exterior            clip_004                          0.324   
entrevista, feria               clip_016                          0.173   
entrevista, interior            clip_022                          0.274   
entrevista, manifestación       clip_005                          0.236   
noticiero de archivo con música clip_023                          0.056   
                                clip_025                          0.014   

modelo                                    faster-whisper small (CPU, base)  
condicion                       id                                          
entrevista, calle               clip_011                             0.318  
entrevista, calle (Cusco)       clip_009                             0.438  
entrevista, centro comercial    clip_018                             0.180  
                                clip_019                             0.083  
entrevista, exterior            clip_004                             0.431  
entrevista, feria               clip_016                             0.281  
entrevista, interior            clip_022                             0.315  
entrevista, manifestación       clip_005                             0.743  
noticiero de archivo con música clip_023                             0.360  
                                clip_025                             0.014

In [12]:
mejor_ocr = resumen_ocr["CER medio"].idxmin()
fig = px.bar(cer_docs.groupby(["modelo", "tipo"], as_index=False).cer.mean(), x="tipo", y="cer", color="modelo", barmode="group",
             category_orders={"tipo": ["impreso", "foto", "manuscrito"]}, labels={"cer": "CER medio"},
             title=(f"{mejor_ocr} tiene el menor CER ({resumen_ocr.loc[mejor_ocr, 'CER medio']:.2f}); en manuscritos ningún "
                    f"OCR baja de {resumen_ocr['manuscrito'].min():.2f}"))
fig.update_layout(title_font_size=14)
fig.show()

In [13]:
mejor_asr = resumen_asr["WER medio"].idxmin()
fig = px.scatter(resumen_asr.reset_index(), x="RTF medio", y="WER medio", text="modelo", log_x=True,
                 title=(f"{mejor_asr} logra el menor WER ({resumen_asr.loc[mejor_asr, 'WER medio']:.2f}) con un RTF de "
                        f"{resumen_asr.loc[mejor_asr, 'RTF medio']:.2f}"))
fig.update_traces(textposition="top center", marker_size=12)
fig.update_layout(title_font_size=14)
fig.show()

## 5. Casos difíciles

**Manuscritos**: cartas del siglo XIX y comienzos del XX, con letra cursiva, tinta corrida y papel manchado. **Audio con
ruido o música**: entrevistas en la calle, en una manifestación o en una feria, y noticieros de archivo con música de
fondo. Los ejemplos de abajo usan los textos ya seudonimizados de los CSV.

El análisis de quechua de Huaylas con MMS (opcional) no se hizo: el corpus no tiene audio en quechua y la consigna pide
no recolectar datos nuevos.

In [14]:
manus = cer_docs[cer_docs.tipo == "manuscrito"].pivot_table(index="id", columns="modelo", values="cer")
print(f"CER medio en manuscritos: {manus.mean().round(2).to_dict()}")
display(manus.round(3))
peor = manus.mean(axis=1).idxmax()
d = documentos.set_index("id").loc[peor]
print(f"\n{peor}: lo que lee PaddleOCR (Parte 1) frente a la referencia manual, ya seudonimizados\n")
print("OCR:       ", d.texto_ocr[:300])
print("Referencia:", d.texto_referencia[:300])

CER medio en manuscritos: {'EasyOCR (GPU)': 0.75, 'PaddleOCR Parte 1 (CPU)': 0.37, 'PaddleOCR base (CPU)': 0.37}


modelo,EasyOCR (GPU),PaddleOCR Parte 1 (CPU),PaddleOCR base (CPU)
id,,,
doc_018,0.770,0.498,0.498
doc_019,0.766,0.354,0.354
doc_021,0.633,0.300,0.300
doc_023,0.835,0.347,0.347



doc_018: lo que lee PaddleOCR (Parte 1) frente a la referencia manual, ya seudonimizados

OCR:        Frugillo, 2'9 de pilis 917 of. fori lavii Egue Bannnco. hosvsumente coumovid bé se alut carth, eugos timins le ageadyeo de vesas. [PERSONA_141] me hr Aeanimad nenchio,n setis dias un gue me son tin tan anal, fs agi kay guimes ome alacion coe turt rudiza...! bil gnios, sein Ege se; gti a y o. ome  k 
Referencia: Trujillo, 29 de julio de 1917. Sr [PERSONA_242]. Honestamente conmovido lei su atenta carta cuyos términos le agradezco de veras. Ella me ha reanimado mucho, en estos días en que me sentía tan mal; fue aquí hoy quienes me atacan con tanta rudeza….. mil gracias, señor [PERSONA_243][ilegible] , su gen


In [15]:
ruido = wer_clips.pivot_table(index=["condicion", "id"], columns="modelo", values="wer")
display(ruido.round(3))
por_condicion = wer_clips.assign(grupo=np.where(wer_clips.condicion.str.contains("noticiero"), "noticiero con música",
                                                np.where(wer_clips.condicion.str.contains("interior|exterior"), "entrevista tranquila",
                                                         "entrevista con ruido de calle o público")))
display(por_condicion.pivot_table(index="grupo", columns="modelo", values="wer").round(3))

modelo                                    Qwen3-ASR 0.6B (GPU)  \
condicion                       id                               
entrevista, calle               clip_011                 0.209   
entrevista, calle (Cusco)       clip_009                 0.371   
entrevista, centro comercial    clip_018                 0.286   
                                clip_019                 0.100   
entrevista, exterior            clip_004                 0.363   
entrevista, feria               clip_016                 0.194   
entrevista, interior            clip_022                 0.268   
entrevista, manifestación       clip_005                 0.336   
noticiero de archivo con música clip_023                 0.146   
                                clip_025                 0.099   

modelo                                    faster-whisper large-v3 (GPU)  \
condicion                       id                                        
entrevista, calle               clip_011                          0.264   
entrevista, calle (Cusco)       clip_009                          0.286   
entrevista, centro comercial    clip_018                          0.188   
                                clip_019                          0.075   
entrevista, exterior            clip_004                          0.324   
entrevista, feria               clip_016                          0.173   
entrevista, interior            clip_022                          0.274   
entrevista, manifestación       clip_005                          0.236   
noticiero de archivo con música clip_023                          0.056   
                                clip_025                          0.014   

modelo                                    faster-whisper small (CPU, base)  
condicion                       id                                          
entrevista, calle               clip_011                             0.318  
entrevista, calle (Cusco)       clip_009                             0.438  
entrevista, centro comercial    clip_018                             0.180  
                                clip_019                             0.083  
entrevista, exterior            clip_004                             0.431  
entrevista, feria               clip_016                             0.281  
entrevista, interior            clip_022                             0.315  
entrevista, manifestación       clip_005                             0.743  
noticiero de archivo con música clip_023                             0.360  
                                clip_025                             0.014

modelo,Qwen3-ASR 0.6B (GPU),faster-whisper large-v3 (GPU),"faster-whisper small (CPU, base)"
grupo,,,
entrevista con ruido de calle o público,0.249,0.203,0.340
entrevista tranquila,0.315,0.299,0.373
noticiero con música,0.122,0.035,0.187


## 6. Extracción estructurada con LLM

Qwen2.5-3B-Instruct corre en local (GPU) sobre los textos **seudonimizados** de los 25 documentos y los 24 clips, así
que ningún nombre real sale de los CSV publicados. El prompt pide un JSON con campos fijos por modalidad y `null` cuando
el dato no está en el texto. Se usa decodificación determinista (sin muestreo). Los textos de más de 4000 caracteres
(los diarios) se recortan a sus primeros 4000.

In [16]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_LLM = CONFIGS["llm"]["modelo"]
CAMPOS = {
    "documento": {"modalidad": "documento", "tema": "tema principal en pocas palabras", "resumen": "una o dos oraciones",
                  "tipo_documento": "carta, diario, placa, proclama, oficio u otro", "emisor": "quién firma o publica",
                  "destinatario": "a quién va dirigido", "fecha": "fecha tal como aparece", "lugar": "lugar de emisión",
                  "monto": "cantidad de dinero tal como aparece"},
    "audio": {"modalidad": "audio", "tema": "tema principal en pocas palabras", "resumen": "una o dos oraciones",
              "numero_hablantes": "número entero de personas que hablan, contando las marcas [HABLANTE_n]",
              "citas_clave": "lista de 1 a 3 frases copiadas literalmente del texto", "lugar": "lugar mencionado donde ocurre"},
}
PROMPT = """Eres un asistente que extrae datos de textos peruanos obtenidos por OCR o transcripción automática.
El texto puede tener errores de reconocimiento y marcadores como [PERSONA_3] en lugar de nombres reales: cópialos tal cual.
Devuelve SOLO un objeto JSON con exactamente estas claves:
{campos}
Reglas: usa únicamente información presente en el texto; si un dato no aparece, usa null; no inventes ni completes datos;
escribe en español.

Texto ({modalidad}):
\"\"\"{texto}\"\"\""""

tok = llm = None


def extraer(caso):
    global tok, llm
    if llm is None:
        tok = AutoTokenizer.from_pretrained(MODELO_LLM)
        llm = AutoModelForCausalLM.from_pretrained(MODELO_LLM, dtype=torch.bfloat16).to("cuda")
    modalidad, texto = caso
    campos = "\n".join(f'- "{k}": {v}' for k, v in CAMPOS[modalidad].items())
    mensajes = [{"role": "user", "content": PROMPT.format(campos=campos, modalidad=modalidad, texto=texto[:4000])}]
    entrada = tok.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    with torch.inference_mode():
        salida = llm.generate(**entrada, max_new_tokens=512, do_sample=False)
    return tok.decode(salida[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True)


VACIOS = {"", "null", "none", "n/a", "no especificado", "no especificada", "desconocido", "desconocida", "no se menciona",
          "no aplica", "ninguno", "ninguna"}


def a_json(respuesta, modalidad):
    """Primer objeto JSON de la respuesta, con solo las claves pedidas. Las que faltan y los textos de relleno
    ("no especificado") quedan en null, que es lo que pide el prompt."""
    m = re.search(r"\{.*\}", respuesta, re.S)
    try:
        datos = json.loads(m.group()) if m else {}
    except json.JSONDecodeError:
        datos = {}
    vacio = lambda v: isinstance(v, str) and v.strip().lower() in VACIOS
    campos = {k: None if vacio(datos.get(k)) else datos.get(k) for k in CAMPOS[modalidad]}
    return campos | {"modalidad": modalidad}, bool(datos)


casos = ([(i, "documento", t) for i, t in zip(documentos.id, documentos.texto_ocr)] +
         [(i, "audio", t) for i, t in zip(transcripciones.id, transcripciones.texto_asr)])
crudas = correr("llm", [c[0] for c in casos], lambda i: extraer(next((m, t) for j, m, t in casos if j == i)))
del llm
_ = gc.collect()
torch.cuda.empty_cache()

filas, invalidas = [], []
for i, modalidad, _ in casos:
    campos, ok = a_json(crudas[i]["texto"], modalidad)
    if not ok:
        invalidas.append(i)
    filas.append({"id": i, "modalidad": modalidad, "campos_json": json.dumps(campos, ensure_ascii=False),
                  "modelo_llm": f"{MODELO_LLM} (local, bf16, sin muestreo)", "fecha": date.today().isoformat()})
extraccion = pd.DataFrame(filas)
extraccion.to_csv(DATOS / "extraccion_llm.csv", index=False)
print(f"datos/extraccion_llm.csv {extraccion.shape} · respuestas sin JSON válido: {invalidas or 'ninguna'}")
print(f"Segundos por extracción (mediana): {np.median([crudas[i]['segundos'] for i, _, _ in casos]):.1f}")
extraccion.head()

datos/extraccion_llm.csv (49, 5) · respuestas sin JSON válido: ninguna
Segundos por extracción (mediana): 5.7


,id,modalidad,campos_json,modelo_llm,fecha
0,doc_001,documento,"{""modalidad"": ""documento"", ""tema"": ""Felicitaciones y nombramiento"", ""resumen...","Qwen/Qwen2.5-3B-Instruct (local, bf16, sin muestreo)",2026-10-26
1,doc_002,documento,"{""modalidad"": ""documento"", ""tema"": ""Expedición militar chilena al Perú"", ""re...","Qwen/Qwen2.5-3B-Instruct (local, bf16, sin muestreo)",2026-10-26
2,doc_003,documento,"{""modalidad"": ""documento"", ""tema"": ""droguería italiana y medicamentos"", ""res...","Qwen/Qwen2.5-3B-Instruct (local, bf16, sin muestreo)",2026-10-26
3,doc_004,documento,"{""modalidad"": ""documento"", ""tema"": ""conflicto choferes vs. empresa"", ""resume...","Qwen/Qwen2.5-3B-Instruct (local, bf16, sin muestreo)",2026-10-26
4,doc_005,documento,"{""modalidad"": ""documento"", ""tema"": ""Guerra y conflicto en Nueva Granada"", ""r...","Qwen/Qwen2.5-3B-Instruct (local, bf16, sin muestreo)",2026-10-26


In [17]:
# Cobertura de campos: qué tan seguido el modelo encuentra cada dato
cobertura = (pd.DataFrame([json.loads(c) for c in extraccion.campos_json]).drop(columns="modalidad")
             .assign(modalidad=extraccion.modalidad).groupby("modalidad").agg(lambda s: s.notna().mean()).T.round(2))
cobertura

modalidad,audio,documento
tema,1.00,1.00
resumen,1.00,1.00
tipo_documento,0.00,1.00
emisor,0.00,0.96
destinatario,0.00,0.84
fecha,0.00,0.44
lugar,0.71,0.64
monto,0.00,0.00
numero_hablantes,1.00,0.00
citas_clave,1.00,0.00


## 7. Auditoría humana

Se sortean 20 extracciones (10 documentos y 10 audios, semilla 2026) y se revisan a mano contra el texto seudonimizado.

**Criterios**: una extracción es *completamente correcta* si todos sus campos cumplen:

- `tema` y `resumen` describen el contenido real del texto, sin datos que no estén en él.
- Los campos factuales (`emisor`, `destinatario`, `fecha`, `lugar`, `monto`, `tipo_documento`, `numero_hablantes`) son
  correctos o `null` cuando el dato no aparece. Inventar un valor o dejar en `null` un dato claramente presente es error.
- `citas_clave` son frases que de verdad están en la transcripción.
- Un error de lectura que ya venía del OCR o del ASR no se cuenta como error del LLM si lo copió fielmente.

Las verificaciones automáticas de abajo (si las citas aparecen literalmente en el texto, si `numero_hablantes` coincide
con la diarización) ayudan a revisar, pero no reemplazan la auditoría. El resultado se anota a mano en `AUDITORIA`:
`True` si la extracción es completamente correcta, `False` y el motivo si no.

In [18]:
auditar = pd.concat([extraccion[extraccion.modalidad == m].sample(10, random_state=2026) for m in ["documento", "audio"]])
textos = dict(zip(documentos.id, documentos.texto_ocr)) | dict(zip(transcripciones.id, transcripciones.texto_asr))
hablantes = dict(zip(transcripciones.id, transcripciones.n_hablantes))


def chequeos(i, campos):
    avisos = []
    for cita in campos.get("citas_clave") or []:
        if normalizar(cita) not in normalizar(textos[i]):
            avisos.append(f"cita no literal: {cita[:50]!r}")
    if campos.get("numero_hablantes") is not None and i in hablantes and campos["numero_hablantes"] != hablantes[i]:
        avisos.append(f"numero_hablantes {campos['numero_hablantes']} ≠ diarización {hablantes[i]}")
    return "; ".join(avisos)


revision = auditar.assign(campos=auditar.campos_json.map(json.loads))
revision["chequeo_automatico"] = [chequeos(i, c) for i, c in zip(revision.id, revision.campos)]
for r in revision.itertuples():
    print(f"── {r.id} ({r.modalidad}) {('· ' + r.chequeo_automatico) if r.chequeo_automatico else ''}")
    print("Texto:", textos[r.id][:500].replace("\n", " "))
    print("JSON: ", json.dumps(r.campos, ensure_ascii=False))
    print()

── doc_004 (documento) 
Texto: DE RIQUEZAS NACIONA- NO PODEMOS HABLAR LA-TRIBUNA CONTRA EL CIVILISMO LBS, MIENTRAS ESTÈN. EN PODER DE MANOS EXTRANJERIZANTE: EL EXTRANJERAS APRISMO PERUANIZADOR DIARIO APRISTA INFORMATIVO DE LA MAÑANA.—DIRECTOR: [PERSONA_46] y Administración: Camaná No. 540. Telf. 30811. Precio: Cince-centaves no. LIMA, SABADO 16 DE MAYO DE 1931. Número 1 ¿TRIUNFO LA CAUSA DE LOS CHOFERES? Créese aceptado el derecho MAS DE SIETE HORAS SESIONO ¿AMENAZAS? de los autos "colectivos" para AYER EL TRIBUNAL ARBITRAL S
JSON:  {"modalidad": "documento", "tema": "conflicto choferes vs. empresa", "resumen": "El documento denuncia el conflicto entre choferes y la empresa de autobuses Metropolitan Bus, con fallo del Tribunal Arbitral que podría poner a la empresa en quiebra.", "tipo_documento": "oficio", "emisor": "[PERSONA_45]", "destinatario": "Tribunal Arbitral", "fecha": "Sábado 16 de mayo de 1931", "lugar": "Lima", "monto": null}

── doc_016 (documento) 
Texto: Lima 23 De Junto 

In [19]:
# Resultado de la revisión manual: True = completamente correcta; (False, "motivo") si no; None = sin revisar
AUDITORIA = {
    "doc_004": True, "doc_016": True, "doc_019": (False, "resumen no coincide con el contenido del documento"), "doc_017": (False, "resumen no coincide con el contenido del documento"),
    "doc_015": True, "doc_025": (False, "resumen no coincide con el contenido del documento"), "doc_005": True, "doc_009": True, "doc_011": True,
    "doc_018": (False, "resumen no coincide con el contenido del documento"),
    "clip_004": (False, "numero_hablantes incorrecto (LLM 4, diarización 2)"), "clip_017": (False, "numero_hablantes incorrecto (LLM 3, diarización 2)"), "clip_020": (False, "numero_hablantes incorrecto (LLM 4, diarización 2)"),
    "clip_018": (False, "tema y resumen hablan de delincuencia, economía, pobreza y críticas al gobierno, pero la entrevista "
                        "pregunta qué significa el Perú y sus características positivas y negativas; además "
                        "numero_hablantes incorrecto (LLM 3, diarización 2)"),
    "clip_016": (False, "numero_hablantes incorrecto (LLM 4, diarización 2)"), "clip_025": (False, "numero_hablantes incorrecto (LLM 2, diarización 1)"), "clip_005": (False, "numero_hablantes incorrecto (LLM 4, diarización 2)"),
    "clip_009": (False, "numero_hablantes incorrecto (LLM 4, diarización 2)"), "clip_011": (False, "numero_hablantes incorrecto (LLM 3, diarización 2)"),
    "clip_019": True, 
}
assert set(AUDITORIA) == set(auditar.id), "AUDITORIA debe tener exactamente los 20 casos sorteados"

resultado = pd.DataFrame([{"id": i, "modalidad": "documento" if i.startswith("doc") else "audio",
                           "correcta": None if v is None else v is True,
                           "motivo": v[1] if isinstance(v, tuple) else ""} for i, v in AUDITORIA.items()])
revisadas = resultado.dropna(subset=["correcta"])
for m, g in revisadas.groupby("modalidad"):
    print(f"{m}: {int(g.correcta.sum())}/{len(g)} completamente correctas ({g.correcta.mean():.0%})")
if len(revisadas) < len(resultado):
    print(f"Auditoría pendiente: {len(revisadas)} de {len(resultado)} extracciones revisadas.")
else:
    print(f"Total: {int(revisadas.correcta.sum())}/{len(revisadas)} ({revisadas.correcta.mean():.0%})")
display(revisadas[~revisadas.correcta.astype(bool)][["id", "modalidad", "motivo"]])

audio: 1/10 completamente correctas (10%)
documento: 6/10 completamente correctas (60%)
Total: 7/20 (35%)


,id,modalidad,motivo
2,doc_019,documento,resumen no coincide con el contenido del documento
3,doc_017,documento,resumen no coincide con el contenido del documento
5,doc_025,documento,resumen no coincide con el contenido del documento
9,doc_018,documento,resumen no coincide con el contenido del documento
10,clip_004,audio,"numero_hablantes incorrecto (LLM 4, diarización 2)"
11,clip_017,audio,"numero_hablantes incorrecto (LLM 3, diarización 2)"
12,clip_020,audio,"numero_hablantes incorrecto (LLM 4, diarización 2)"
13,clip_018,audio,"tema y resumen hablan de delincuencia, economía, pobreza y críticas al gobie..."
14,clip_016,audio,"numero_hablantes incorrecto (LLM 4, diarización 2)"
15,clip_025,audio,"numero_hablantes incorrecto (LLM 2, diarización 1)"


**Resultado de la auditoría: 7/20 extracciones completamente correctas (35 %)**, 6/10 en documentos y 1/10 en audio.

- **Documentos**: las 4 extracciones incorrectas (`doc_017`, `doc_018`, `doc_019`, `doc_025`) son manuscritos. Con un
  CER de 0.3 a 0.5, el texto que recibe el LLM es casi ilegible y el modelo rellena con contenido que no está: en
  `doc_019` inventa un "Comandante Somalias" y un "incidente militar". En impresos y fotos legibles acierta.
- **Audio**: el LLM falla `numero_hablantes` en 9 de 10 clips. Cuenta los turnos marcados con `[HABLANTE_n]`, no las
  personas distintas, aunque el prompt pide personas. En `clip_018` además resume lo que se critica en vez de lo que se
  pregunta. El tema, el resumen y las citas sí suelen ser correctos.
- **Qué cambiar en un entorno real**: tomar `numero_hablantes` de la diarización y no del LLM, y no pedir resumen a
  textos con confianza de OCR baja (los manuscritos tienen 0.51 a 0.74) o marcarlos para revisión humana.

## 8. Comparativa y decisión

In [20]:
DISPOSITIVO = {"PaddleOCR base (CPU)": "CPU", "PaddleOCR Parte 1 (CPU)": "CPU", "EasyOCR (GPU)": "GPU (RTX 4060)",
               "faster-whisper small (CPU, base)": "CPU", "faster-whisper large-v3 (GPU)": "GPU (RTX 4060)",
               "Qwen3-ASR 0.6B (GPU)": "GPU (RTX 4060)"}
ROL = {"PaddleOCR base (CPU)": "línea base", "PaddleOCR Parte 1 (CPU)": "mejora (preprocesamiento)", "EasyOCR (GPU)": "alternativa",
       "faster-whisper small (CPU, base)": "línea base", "faster-whisper large-v3 (GPU)": "mejora (modelo grande)",
       "Qwen3-ASR 0.6B (GPU)": "alternativa"}
tabla = pd.concat([
    resumen_ocr.reset_index().rename(columns={"CER medio": "error medio", "segundos por página": "tiempo"})
    .assign(modalidad="documentos (CER)", tiempo=lambda d: d.tiempo.map(lambda s: f"{s:.1f} s/página" if pd.notna(s) else "sin medir")),
    resumen_asr.reset_index().rename(columns={"WER medio": "error medio"})
    .assign(modalidad="audio (WER)", tiempo=lambda d: d["RTF medio"].map(lambda r: f"RTF {r:.2f}")),
])[["modalidad", "modelo", "error medio", "impreso", "foto", "manuscrito", "tiempo"]]
tabla["rol"] = tabla.modelo.map(ROL)
tabla["dispositivo"] = tabla.modelo.map(DISPOSITIVO)
tabla.round(3).reset_index(drop=True)

,modalidad,modelo,error medio,impreso,foto,manuscrito,tiempo,rol,dispositivo
0,documentos (CER),EasyOCR (GPU),0.439,0.206,0.255,0.751,2.1 s/página,alternativa,GPU (RTX 4060)
1,documentos (CER),PaddleOCR Parte 1 (CPU),0.323,0.055,0.522,0.375,sin medir,mejora (preprocesamiento),CPU
2,documentos (CER),PaddleOCR base (CPU),0.255,0.055,0.297,0.375,29.4 s/página,línea base,CPU
3,audio (WER),Qwen3-ASR 0.6B (GPU),0.237,NaN,NaN,NaN,RTF 0.30,alternativa,GPU (RTX 4060)
4,audio (WER),faster-whisper large-v3 (GPU),0.189,NaN,NaN,NaN,RTF 0.08,mejora (modelo grande),GPU (RTX 4060)
5,audio (WER),"faster-whisper small (CPU, base)",0.316,NaN,NaN,NaN,RTF 0.28,línea base,CPU


In [21]:
# Sensibilidad: cuánto del WER viene de muletillas que la referencia transcribe y Whisper suele omitir
MULETILLAS = r"\b(eh+|uhm+|mm+|ehm+)\b"
sin_muletillas = lambda t: re.sub(MULETILLAS, " ", normalizar(t))
sens = pd.DataFrame([{"modelo": m, "id": i, "wer": wer(referencias[i], r[i]["texto"]),
                      "wer_sin_muletillas": jiwer.wer(sin_muletillas(referencias[i]), sin_muletillas(r[i]["texto"]) or " ")}
                     for m, r in ASR.items() for i in muestra_clips.id])
sens.groupby("modelo")[["wer", "wer_sin_muletillas"]].mean().round(3)

,wer,wer_sin_muletillas
modelo,,
Qwen3-ASR 0.6B (GPU),0.237,0.227
faster-whisper large-v3 (GPU),0.189,0.175
"faster-whisper small (CPU, base)",0.316,0.305


### Decisión

**Documentos: PaddleOCR con su configuración por defecto (sin CLAHE).** Tiene el menor CER medio (0.26) y es el mejor
en impresos (0.05) y manuscritos (0.37). El preprocesamiento de la Parte 1 no ayudó: el CLAHE subió el CER de las
fotos de 0.30 a 0.52, porque realza la textura de la piedra y el metal de las placas tanto como las letras. EasyOCR es
14 veces más rápido en GPU (2 s por página contra 29 s de PaddleOCR en CPU), pero su CER medio es 0.44. Para un
entorno real conviene probar PaddleOCR en GPU (aquí corrió en CPU): debería bajar el tiempo con los mismos modelos.

**Audio: faster-whisper large-v3 en GPU.** Es el más preciso (WER 0.19) y también el más rápido (RTF 0.08: un minuto
de audio en unos 5 s). Su costo es la GPU: necesita unos 4 GB de VRAM en `float16`. Sin GPU, la línea base
(faster-whisper small en CPU) sigue siendo usable en tiempo real (RTF 0.28), pero comete un 67 % más de errores (WER
0.32) y se cae en la manifestación (`clip_005`, 0.74). Qwen3-ASR 0.6B queda en medio (WER 0.24) sin ser más rápido que
large-v3, así que no compensa cambiar.

### Reflexión por tipo

**Documentos**

- **Impresos (mejor desempeño, CER 0.04–0.06).** Tipografía regular y buen contraste. Los errores son sobre todo la
  ortografía antigua ("exércitos", "quales"), sellos y anotaciones al margen que el OCR lee y la referencia no.
- **Fotos (muy variable, 0.01–0.82).** Una carta mecanografiada fotografiada (`doc_011`) sale casi perfecta, pero la
  placa de la casa Elguera (`doc_013`) se lee mal: letras en relieve sobre metal con poco contraste, y la "V" latina en
  lugar de "U" ("PERV", "CIVDAD") que el modelo no espera.
- **Manuscritos (peor desempeño, 0.30–0.50).** La letra cursiva del siglo XIX, las abreviaturas y la tinta corrida
  están fuera de lo que PP-OCRv6 y EasyOCR aprendieron, que es sobre todo texto impreso. Un CER de 0.3 deja el texto
  ilegible para un lector: aquí hace falta un modelo de reconocimiento de escritura a mano (HTR) o transcripción humana.

**Audio**

- **Noticieros de archivo (mejor, WER 0.01–0.06 con large-v3).** Narración de estudio, dicción cuidada y un solo
  locutor. La música de fondo no afecta a large-v3, pero sí a small (0.36 en `clip_023`).
- **Entrevistas en la calle o con público (0.08–0.29 con large-v3).** Ruido de tráfico, voces superpuestas y frases cortadas. La
  peor es la manifestación con small, porque la multitud tapa la voz.
- **Entrevistas tranquilas (0.27–0.32 con large-v3).** Lo que sube el WER no es el ruido sino el habla espontánea: repeticiones
  ("es es es es"), interrupciones ("oye, hola papá") y nombres propios. Además, la referencia transcribe muletillas
  ("eh", "uhm") que Whisper omite, pero eso explica poco: sin muletillas el WER baja apenas 0.01 en los tres modelos.